# 第12回　不均衡データと ROC
***
> **前提**: 第6回の評価指標を発展させ，クラス不均衡への対処と ROC/PR 曲線を学びます。

## 目次
1. 不均衡データの生成
2. class_weight
3. ROC 曲線と AUC
4. 適合率-再現率曲線

---

## この回で学ぶこと

### クラス不均衡が問題になる場面

現実の分類問題では，クラスの数が極端に偏っていることが多い：

| 問題 | クラス比率 |
|---|---|
| クレジットカード不正検知 | 正常 99.8% / 不正 0.2% |
| 医療診断（稀な疾患） | 陰性 99% / 陽性 1% |
| 製品不良品検出 | 良品 99% / 不良品 1% |

このような状況で「全部多数クラスと予測」すると正解率 99.8% になるが，**不正を一件も検知できていない**ので全く役に立たないモデルだ。

### 混同行列（Confusion Matrix）の理解

混同行列は分類モデルの結果を4つの分類で整理する：

```
                予測: Negative  予測: Positive
実際: Negative      TN               FP
実際: Positive      FN               TP
```

- **TP（True Positive）**: 陽性を陽性と正しく予測
- **TN（True Negative）**: 陰性を陰性と正しく予測
- **FP（False Positive）**: 陰性を陽性と誤予測（偽陽性）→ 「誤報」
- **FN（False Negative）**: 陽性を陰性と誤予測（偽陰性）→ 「見逃し」

### 各評価指標の意味

- **適合率（Precision）** = TP / (TP + FP)：「陽性と予測したうち実際に陽性の割合」→ **誤報を減らしたい**時に重視
- **再現率（Recall）** = TP / (TP + FN)：「実際の陽性のうち正しく予測した割合」→ **見逃しを減らしたい**時に重視（医療では特に重要）
- **F1スコア** = 2 × Precision × Recall / (Precision + Recall)：両者の調和平均

### ROC 曲線と AUC

ROC 曲線は**分類閾値を 0〜1 で変化させた時**の「偽陽性率（FPR）vs 真陽性率（TPR）」をプロットしたものだ。

- 完璧なモデル：左上の角に近い曲線
- ランダムな予測：対角線（AUC = 0.5）
- **AUC（Area Under the Curve）**：ROC 曲線の下の面積（0〜1）。1に近いほど良い

### PR 曲線が重要な場面

不均衡データでは ROC 曲線が「楽観的に見える」場合がある（TN が多いため FPR が小さくなりやすい）。**適合率-再現率曲線（PR曲線）** は少数クラスに注目した評価に向いている。

> **卒業研究での指針**: 不均衡データを扱う場合は，正解率だけでなく F1スコア，AUC，PR-AUC を報告することが学術的に求められる。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay,
    auc,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split


## 問題1　不均衡データの生成と確認
***

### `make_classification` のパラメータ解説

`make_classification` は人工的な分類データを生成する。今回のパラメータ：
- `n_samples=2000`：2000件のデータ
- `n_features=20`：20個の特徴量
- `weights=[0.95, 0.05]`：**クラス0が95%，クラス1が5%** という不均衡設定
- `random_state=0`：再現性のため固定

### 不均衡度の確認が最初のステップ

データを受け取ったら，まず**クラスの分布を確認**することが重要だ。不均衡比率によって対処法が変わる：
- 5:1 程度：`class_weight="balanced"` で対処可能（今回）
- 100:1 以上：オーバーサンプリング（SMOTE）やアンダーサンプリングが必要

### 課題

`make_classification(n_samples=2000, n_features=20, weights=[0.95, 0.05], random_state=0)` で不均衡な2値分類データを生成してください。

train/test（80:20）に分割し，訓練データにおけるクラス0・クラス1の**サンプル数**と**比率**を出力してください。

> **確認ポイント**: `weights=[0.95, 0.05]` 設定でどれだけ偏ったデータになるか確認しよう。`pd.Series(y_train).value_counts()` で簡単に確認できる。


In [ ]:
# 不均衡データの生成
# ここにあなたのコードを書いてください


## 問題2　class_weight による不均衡データ対処
***

### `class_weight="balanced"` の仕組み

`class_weight="balanced"` を指定すると，scikit-learn はクラスの出現頻度に反比例した重みを自動計算する：

```
クラスiの重み = 全サンプル数 / (クラス数 × クラスiのサンプル数)

例: クラス0が1900件，クラス1が100件，計2000件の場合
  クラス0の重み = 2000 / (2 × 1900) ≈ 0.53
  クラス1の重み = 2000 / (2 × 100) = 10.0
```

少数クラス（クラス1）の誤分類に大きなペナルティをかけることで，モデルが少数クラスを積極的に検出するようになる。

### Precision と Recall のトレードオフを理解する

`class_weight="balanced"` を使うと：
- 再現率（クラス1の見逃し率が下がる）↑
- 適合率（クラス1と予測したうちの正解率）↓ （になることが多い）

このトレードオフは**どちらのエラーがより重大か**によって判断する：
- 癌の診断：見逃し（FN）が致命的 → 再現率を優先
- スパムフィルター：誤判定（FP）が嫌 → 適合率を優先

### 課題

`LogisticRegression(max_iter=1000)` を **class_weight なし**で学習し，テストデータの混同行列と `classification_report` を出力してください。

次に `class_weight="balanced"` を指定したモデルでも同様に評価し，**クラス1の再現率（Recall）の変化**を確認してください。

> **考えてみよう**: class_weight なしのモデルは，クラス1をほとんど見逃しているのではないか？混同行列の FN（偽陰性）の数を確認しよう。


In [ ]:
# class_weight の比較
# ここにあなたのコードを書いてください


## 問題3　ROC 曲線と AUC
***

### ROC 曲線の読み方

ROC 曲線は「閾値を変化させたとき，どれだけ上手く分類できるか」を可視化する：

```
縦軸：真陽性率（TPR = 再現率）= TP / (TP + FN)
横軸：偽陽性率（FPR）= FP / (FP + TN)

・左上の角（TPR=1, FPR=0）が理想
・対角線（y=x）はランダムな予測（AUC=0.5）
```

**AUC の解釈**:
- ランダムに選んだ陽性サンプルのスコアが，ランダムに選んだ陰性サンプルのスコアより高い確率 = AUC

つまり AUC=0.9 なら，「陽性と陰性をランダムに1件ずつ選んだとき，モデルが陽性を高くスコアリングする確率が90%」という意味だ。

### class_weight なし vs あり の AUC 比較

今回は `class_weight="balanced"` のモデルの ROC 曲線を描く。**AUC はクラスの不均衡に比較的ロバスト**（頑健）なため，正解率が全く役に立たない不均衡データでも，モデルの性能を適切に評価できる。

### 課題

`class_weight="balanced"` のモデルについて，テストデータの **ROC 曲線**を描画し，**AUC** を出力してください。

#### Hints
- **方法①（簡単）**: `RocCurveDisplay.from_estimator` にモデルとテストデータを渡すだけで曲線が描ける
- **方法②（手動）**: `model.predict_proba` の返り値は `(n_samples, n_classes)` の行列。クラス1の確率は1列目（インデックス1）にある。それを `roc_curve` に渡すと `fpr, tpr, thresholds` が得られる
- AUC は `auc(fpr, tpr)` で計算できる

In [ ]:
# ROC 曲線と AUC
# ここにあなたのコードを書いてください


## 問題4　適合率-再現率曲線（PR曲線）
***

### なぜ不均衡データでは PR 曲線が重要か

不均衡データ（クラス0が95%）では ROC 曲線が「楽観的に見える」問題がある：

```
TN が非常に多い
→ FPR = FP / (FP + TN) が自動的に小さくなりやすい
→ ROC 曲線が左上に寄る
→ AUC が実際より高く見える
```

PR 曲線は TN を使わないため，少数クラス（クラス1）の検出性能をよりシビアに評価できる。

### PR 曲線の読み方

```
縦軸：適合率（Precision）= TP / (TP + FP)
横軸：再現率（Recall）  = TP / (TP + FN)

・右上（Precision=1, Recall=1）が理想
・曲線の下の面積（PR-AUC または Average Precision）が高いほど良い
・不均衡データでは PR-AUC のベースラインは「少数クラスの比率」（今回は0.05）
```

### 課題

同モデルについて **適合率-再現率曲線（PR曲線）**を描画してください。

グラフには「ランダムな予測のベースライン（少数クラスの比率 = 0.05）」を水平点線で追加してください。

> **考えてみよう**: PR 曲線の面積（PR-AUC）と ROC の AUC を比べると，どちらのモデル評価が「厳しい」値になるか？

#### Hints
- `precision_recall_curve` の引数は `(正解ラベル, 予測確率)` で、返り値は `precision, recall, thresholds` の3つ
- PR曲線は **recall を横軸、precision を縦軸**にプロットする（ROC 曲線とは軸が異なる点に注意）
- 不均衡データのランダム予測のベースラインは「少数クラスの比率」（今回は約 0.05）なので、それを水平線として追加すると比較しやすい

In [ ]:
# PR 曲線
# ここにあなたのコードを書いてください
